In [2]:
// Parameters
let batch_mode: &str = "true";


# Complexity-03f — La chaîne des réductions : de 3SUM au triangle lopsided

**Série Complexity — accrétion de 03** · Public : **Recherche** · Hommage : **Josh Alman & Virginia Vassilevska Williams** (*Truly Subquadratic 3SUM and Truly Subcubic APSP via Triangles in Sparse Lopsided Graphs*, arXiv 2610.06783, 10/2026 — copie archivée au gisement `Bibliographie IA/Automata/Complexity Theory and Hardness`)

> « The web of reductions that was interpreted to imply hardness is exactly what turns a single new algorithm into faster algorithms for a myriad of problems at once. »
> — Alman & Vassilevska Williams, introduction (p. 5), notre traduction : *la toile de réductions qu'on interprétait comme une preuve de dureté est exactement ce qui transforme un seul nouvel algorithme en algorithmes plus rapides pour une myriade de problèmes d'un coup.*

**Version Rust** : la bibliothèque standard (`std`) seule — aucun crate externe, aucune dépendance à installer. Le générateur aléatoire (MT19937), le rationnel et la mesure du temps sont écrits dans le carnet.

**Navigation** : [Index](README.md) | [Version C#](Complexity-03f-ReductionsTriangle-Lopsided-CSharp.ipynb) | [Version Python](Complexity-03f-ReductionsTriangle-Lopsided-Python.ipynb)


## 0. Pourquoi ce carnet

Le [03e](Complexity-03e-Barrieres-Exposants-WordRAM-Python.ipynb) a vérifié les cinq claims d'Alman & Vassilevska Williams **là où ils sont énoncés** : la machine word RAM, le témoin fermé, le contrôle quadratique. Il n'a pas montré **comment** un seul algorithme nouveau peut à la fois accélérer 3SUM, APSP, le produit (min,+), la k-Clique de poids nul et trois conjectures de matrices en ligne. La réponse tient en un mot : des **réductions** — celles-là mêmes qui, pendant quinze ans, servaient à prouver que ces problèmes étaient durs.

Ce carnet **marche la chaîne à pieds joints**, sur des instances :

1. les trois hypothèses de la complexité à grains fins et leur toile ;
2. la réduction de 3SUM au **triangle exact** (poids de somme nulle), construite et vérifiée sur corpus ;
3. le maillon **All-Edges Sparse Triangle** (la disjointness d'ensembles hors ligne), sa barrière $m^{4/3}$ et pourquoi les instances produites par les réductions semblaient la coller au mur ;
4. la version **lopsided** — un produit matriciel « fin » $X \in \mathbb{Z}^{N\times D}$, $Y \in \mathbb{Z}^{D\times N}$ avec $D \le N^{1/18}$ — implémentée à l'échelle jouet, entrées voulues contre produit complet ;
5. ce qui survit (SETH, Orthogonal Vectors, les modèles restreints) et ce qui tombe.

**Prérequis** : [03](Complexity-03-HierarchieTemporelle-Python.ipynb) (hiérarchie en temps), [02](Complexity-02-P-NP-Reduction-Python.ipynb) (l'idée de réduction), [03e](Complexity-03e-Barrieres-Exposants-WordRAM-Python.ipynb) (les cinq claims et leur formalisation). Aucune ressource externe : tout est généré et mesuré ici.


In [3]:
// tout le carnet est rejouable a l'identique
const SEED: u32 = 20261111;

// std suffit : MT19937 local (reimplementation de random.Random), rationnel local
// (struct Frac), temps std (std::time::Instant).
println!("Imports OK : std — MT19937 local, rationnel local, temps std — SEED={}", SEED);


Imports OK : std — MT19937 local, rationnel local, temps std — SEED=20261111


## 1. Trois hypothèses et une toile

La complexité à grains fins (*fine-grained complexity*) part d'un constat : pour une famille de problèmes polynomiaux, aucun progrès d'exposant depuis des décennies. Trois hypothèses ont servi de socle (papier, p. 3) :

| Hypothèse | Énoncé (word RAM, mots $O(\log n)$) | Meilleur connu avant 10/2026 |
|---|---|---|
| **3SUM** | $n$ entiers de taille polynomiale exigent $n^{2-o(1)}$ | $O(n^2)$, gagné seulement en polylog |
| **APSP** | plus courts chemins toutes paires exigent $n^{3-o(1)}$ | $O(n^3 / 2^{\Omega(\sqrt{\log n})})$ — sous-polynomial |
| **SETH** | CNF-SAT à $n$ variables, $O(n)$ clauses : pas de $(2-\varepsilon)^n$ | $2^n$ (à $c^n$ près) |

L'outil du domaine est la **réduction à grains fins** : de $A$ vers $B$ relativement à $a(n)$ et $b(n)$ — si $B$ se résout en $O(b(n)^{1-\varepsilon})$, alors $A$ se résout en $O(a(n)^{1-\delta})$. Une réduction de $A$ vers $B$ transmet la dureté de $A$ vers $B$… et devient un **algorithme pour $A$** le jour où $B$ devient facile.

```mermaid
flowchart LR
    SETH["SETH — CNF-SAT<br/>(intacte)"]
    OV["Orthogonal Vectors<br/>(intacte)"]
    TSUM["Classe 3SUM<br/>(geom., conv-3SUM...)"]
    APSPC["Classe APSP<br/>((min,+), Metricity,<br/>Tree Edit Distance...)"]
    KCL["k-Clique de poids nul"]
    ET["Triangle Exact"]
    AEST["All-Edges Sparse Triangle<br/>(barrière m^4/3)"]
    LOP["Lopsided All-Edges<br/>Sparse Triangle<br/>— le moteur du papier"]
    SETH -.->|non réduit| OV
    TSUM --> ET
    APSPC --> ET
    KCL --> ET
    ET --> AEST
    AEST --> LOP
    LOP -.->|nouvel algorithme| TSUM
    LOP -.->|nouvel algorithme| APSPC
    LOP -.->|nouvel algorithme| KCL
```

La flèche pleine descend vers le problème central ; la flèche pointillée est le renversement d'octobre 2026 : le moteur lopsided remonte la toile. SETH et Orthogonal Vectors ne figurent **pas** dans la toile descendante du papier — elles restent intactes.


In [4]:
// Mt19937 : les mots MT de CPython reimplementes, pour reproduire exactement les
// tirages du carnet Python (random.Random == init_by_array([seed])).
struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }  // random.Random(seed), 0 <= seed < 2**32
    fn new_by_array(key: &[u32]) -> Mt19937 {     // init_by_array(key) de CPython
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {                         // i continue la ou la boucle precedente l'a laisse
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {              // random.random() : 53 bits
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 {  // 1 <= k <= 64 ; mots petit-boutistes, le dernier tronque
        if k <= 32 { return (self.next_u32() >> (32 - k)) as u64; }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    fn randbelow(&mut self, n: u64) -> u64 {     // n >= 1 ; rejet sur getrandbits(n.bit_length())
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randint(&mut self, a: i64, b: i64) -> i64 { a + self.randbelow((b - a + 1) as u64) as i64 }
    fn choice<'a, T>(&mut self, seq: &'a [T]) -> &'a T { &seq[self.randbelow(seq.len() as u64) as usize] }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() { let j = self.randbelow(i as u64 + 1) as usize; x.swap(i, j); }
    }
    // ---- ajouts propres a Complexity-03f ----
    fn randrange(&mut self, start: i64, stop: i64) -> i64 {  // randrange(a, b), pas=1
        start + self.randbelow((stop - start) as u64) as i64
    }
    fn randrange1(&mut self, stop: i64) -> i64 {             // randrange(n) = _randbelow(n)
        self.randbelow(stop as u64) as i64
    }
    fn sample_range(&mut self, lo: i64, hi: i64, k: usize) -> Vec<i64> {  // sample(range(lo,hi),k)
        let n = (hi - lo) as u64;
        assert!(k as u64 <= n);
        let mut result = vec![0i64; k];
        let mut setsize: u64 = 21;                            // setsize de CPython
        if k > 5 {
            setsize += 4u64.pow((((k as f64) * 3.0).log(4.0)).ceil() as u32);
        }
        if n <= setsize {
            let mut pool: Vec<i64> = (lo..hi).collect();
            for i in 0..k {
                let j = self.randbelow(n - i as u64) as usize;
                result[i] = pool[j];
                pool[j] = pool[n as usize - i - 1];
            }
        } else {
            let mut selected: std::collections::HashSet<u64> = std::collections::HashSet::new();
            for i in 0..k {
                let mut j = self.randbelow(n);
                while selected.contains(&j) { j = self.randbelow(n); }
                selected.insert(j);
                result[i] = lo + j as i64;
            }
        }
        result
    }
}

// Booleens a la maniere Python (True / False).
fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

/// Decide 3SUM (trois indices distincts, somme nulle) en O(n^2).
/// Retourne (trouve, triple, operations).
fn trois_sommes_quadratique(nombres: &[i64]) -> (bool, Option<(i64, i64, i64)>, i64) {
    let mut s = nombres.to_vec();
    s.sort();
    let n = s.len();
    let mut ops: i64 = 0;
    for i in 0..n.saturating_sub(2) {
        let cible = -s[i];
        let mut lo = i + 1;
        let mut hi = n.saturating_sub(1);
        while lo < hi {
            ops += 1;
            let deux = s[lo] + s[hi];
            if deux == cible {
                return (true, Some((s[i], s[lo], s[hi])), ops);
            }
            if deux < cible {
                lo += 1;
            } else {
                hi -= 1;
            }
        }
    }
    (false, None, ops)
}

/// Instance 3SUM a n entiers dans [-borne, borne].
fn instance_3sum(n: usize, avec_solution: bool, borne: i64, rng: &mut Mt19937) -> Vec<i64> {
    let mut vals = rng.sample_range(-borne, borne + 1, n);
    if avec_solution {
        let ab = rng.sample_range(1, borne / 2, 2);
        let (a, b) = (ab[0], ab[1]);
        vals[0] = a;
        vals[1] = b;
        vals[2] = -(a + b);
    } else {
        // corpus negatif : on force un trou autour de zero parite (heuristique,
        // l'exhaustivite n'est pas requise : le controle est fait par oracle lent)
    }
    vals
}

/// Oracle O(n^3) : triple distinct de somme nulle.
fn oracle_3sum_lent(nombres: &[i64]) -> (bool, Option<(i64, i64, i64)>) {
    let n = nombres.len();
    for i in 0..n {
        for j in (i + 1)..n {
            for k in (j + 1)..n {
                if nombres[i] + nombres[j] + nombres[k] == 0 {
                    return (true, Some((nombres[i], nombres[j], nombres[k])));
                }
            }
        }
    }
    (false, None)
}

let n_demo: i64 = 60;
let mut rng = Mt19937::new(20261111);
let inst_oui = instance_3sum(n_demo as usize, true, 500, &mut rng);
// tous 1 mod 3 et > 0 : somme de trois ne fait jamais 0
let inst_non: Vec<i64> = (0..n_demo).map(|k| 3 * k + 1).collect();
assert!(!oracle_3sum_lent(&inst_non).0);

let (t_oui, tri_oui, ops_oui) = trois_sommes_quadratique(&inst_oui);
let (t_non, _tri_non, ops_non) = trois_sommes_quadratique(&inst_non);
let triple_txt = match tri_oui {
    Some((a, b, c)) => format!("({}, {}, {})", a, b, c),
    None => "None".to_string(),
};
println!("Instance AVEC solution : trouve={}, triple={}, ops={}", py_bool(t_oui), triple_txt, ops_oui);
println!("Instance SANS solution (oracle lent d'accord) : trouve={}, ops={}", py_bool(t_non), ops_non);
println!("Rapport ops_non / (n^2/2) = {:.3} — le cadrage double balayage", ops_non as f64 / ((n_demo * n_demo) as f64 / 2.0));


Instance AVEC solution : trouve=True, triple=(-447, 127, 320), ops=101


Instance SANS solution (oracle lent d'accord) : trouve=False, ops=1711


Rapport ops_non / (n^2/2) = 0.951 — le cadrage double balayage


### Lecture du résultat — le mur quadratique, mesuré

L'instance **sans** solution coûte exactement $(n-1)(n-2)/2$ comparaisons de paires dans ce double balayage, soit asymptotiquement $n^2/2$. C'est la mesure de cet algorithme sur cette entrée, non une preuve de la conjecture 3SUM. La borne $O(n^{1.9992})$ est citée, pas exécutée ici (voir [03e](Complexity-03e-Barrieres-Exposants-WordRAM-Python.ipynb)).


## 2. De 3SUM au triangle exact

**Triangle Exact** : un graphe tripartite à poids entiers sur les arêtes ; existe-t-il un triangle dont les trois poids somment à zéro ?

La construction (forme linéaire, maillon classique de la chaîne — [VW10, VW18] dans le papier, p. 4) part du 3SUM **triparti** : trois ensembles $A, B, C$, cherche-t-on $a+b+c=0$ à travers les trois ? Alors le graphe a trois parties copiant $A, B, C$, et les poids :

$$w(u^A_a, u^B_b) = a, \qquad w(u^B_b, u^C_c) = b, \qquad w(u^C_c, u^A_a) = c$$

Chaque triangle $(a,b,c)$ porte la somme $a+b+c$ : le triangle de poids nulle somme **est** le triple nul. Le graphe est complet tripartite — le triangle exact parcourt tous les triples, y compris ceux qui réutilisent la même *valeur* : c'est pourquoi on part du 3SUM triparti (indices disjoints par construction).

Le pont depuis le 3SUM mono-ensemble (calcul du carnet, pont pédagogique) : colorier les $n$ éléments en trois couleurs $A, B, C$ ; un triple nul d'indices distincts est bien colorié avec probabilité $6/27 = 2/9$ (chaque élément reçoit l'une des 3 couleurs indépendamment ; les 6 permutations bijectives conviennent). $r$ coloriages indépendants ratent le triple avec probabilité $(7/9)^r$.


In [5]:
// Mt19937 : les mots MT de CPython reimplementes, pour reproduire exactement les
// tirages du carnet Python (random.Random == init_by_array([seed])).
struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }  // random.Random(seed), 0 <= seed < 2**32
    fn new_by_array(key: &[u32]) -> Mt19937 {     // init_by_array(key) de CPython
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {                         // i continue la ou la boucle precedente l'a laisse
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {              // random.random() : 53 bits
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 {  // 1 <= k <= 64 ; mots petit-boutistes, le dernier tronque
        if k <= 32 { return (self.next_u32() >> (32 - k)) as u64; }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    fn randbelow(&mut self, n: u64) -> u64 {     // n >= 1 ; rejet sur getrandbits(n.bit_length())
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randrange(&mut self, start: i64, stop: i64) -> i64 {  // randrange(a, b), pas=1
        start + self.randbelow((stop - start) as u64) as i64
    }
    fn randrange1(&mut self, stop: i64) -> i64 {             // randrange(n) = _randbelow(n)
        self.randbelow(stop as u64) as i64
    }
    fn sample_range(&mut self, lo: i64, hi: i64, k: usize) -> Vec<i64> {  // sample(range(lo,hi),k)
        let n = (hi - lo) as u64;
        assert!(k as u64 <= n);
        let mut result = vec![0i64; k];
        let mut setsize: u64 = 21;                            // setsize de CPython
        if k > 5 {
            setsize += 4u64.pow((((k as f64) * 3.0).log(4.0)).ceil() as u32);
        }
        if n <= setsize {
            let mut pool: Vec<i64> = (lo..hi).collect();
            for i in 0..k {
                let j = self.randbelow(n - i as u64) as usize;
                result[i] = pool[j];
                pool[j] = pool[n as usize - i - 1];
            }
        } else {
            let mut selected: std::collections::HashSet<u64> = std::collections::HashSet::new();
            for i in 0..k {
                let mut j = self.randbelow(n);
                while selected.contains(&j) { j = self.randbelow(n); }
                selected.insert(j);
                result[i] = lo + j as i64;
            }
        }
        result
    }
}

/// Instance 3SUM a n entiers dans [-borne, borne].
fn instance_3sum(n: usize, avec_solution: bool, borne: i64, rng: &mut Mt19937) -> Vec<i64> {
    let mut vals = rng.sample_range(-borne, borne + 1, n);
    if avec_solution {
        let ab = rng.sample_range(1, borne / 2, 2);
        let (a, b) = (ab[0], ab[1]);
        vals[0] = a;
        vals[1] = b;
        vals[2] = -(a + b);
    } else {
        // corpus negatif : on force un trou autour de zero parite (heuristique,
        // l'exhaustivite n'est pas requise : le controle est fait par oracle lent)
    }
    vals
}

/// Oracle O(n^3) : triple distinct de somme nulle.
#[allow(dead_code)]
fn oracle_3sum_lent(nombres: &[i64]) -> (bool, Option<(i64, i64, i64)>) {
    let n = nombres.len();
    for i in 0..n {
        for j in (i + 1)..n {
            for k in (j + 1)..n {
                if nombres[i] + nombres[j] + nombres[k] == 0 {
                    return (true, Some((nombres[i], nombres[j], nombres[k])));
                }
            }
        }
    }
    (false, None)
}

/// Triangle Exact complet tripartite : w(AB)=a, w(BC)=b, w(CA)=c.
struct GrapheTriparti { a: Vec<i64>, b: Vec<i64>, c: Vec<i64> }

fn graphe_depuis_3sum_triparti(a: &[i64], b: &[i64], c: &[i64]) -> GrapheTriparti {
    GrapheTriparti { a: a.to_vec(), b: b.to_vec(), c: c.to_vec() }
}

fn somme_triangle(g: &GrapheTriparti, ia: usize, ib: usize, ic: usize) -> i64 {
    g.a[ia] + g.b[ib] + g.c[ic]  // w(AB)=a, w(BC)=b, w(CA)=c
}

/// Oracle brutal O(|A|.|B|.|C|) : premier triangle de somme nulle.
fn triangle_exact_brut(g: &GrapheTriparti) -> (bool, Option<(i64, i64, i64)>) {
    for ia in 0..g.a.len() {
        for ib in 0..g.b.len() {
            for ic in 0..g.c.len() {
                if somme_triangle(g, ia, ib, ic) == 0 {
                    return (true, Some((g.a[ia], g.b[ib], g.c[ic])));
                }
            }
        }
    }
    (false, None)
}

/// 3SUM mono-ensemble -> 3SUM triparti par 3-coloriage.
fn triparti_par_coloriage(nombres: &[i64], rng: &mut Mt19937) -> (Vec<i64>, Vec<i64>, Vec<i64>) {
    let mut a: Vec<i64> = Vec::new();
    let mut b: Vec<i64> = Vec::new();
    let mut c: Vec<i64> = Vec::new();
    for &v in nombres {
        let coul = rng.randrange1(3);
        if coul == 0 { a.push(v); } else if coul == 1 { b.push(v); } else { c.push(v); }
    }
    (a, b, c)
}

// --- verification d'equivalence sur corpus ---
// cote SANS triple (valeurs 1 mod 3 positives) : aucun coloriage ne doit produire
// de triangle de somme nulle ; cote AVEC triple : 30 coloriages doivent le capturer.
// Sous evcxr rien ne persiste : on rejoue d'abord la cellule 5 (66 mots) sur un
// generateur neuf, sans rien imprimer.
let mut rng = Mt19937::new(20261111);
let _rejeu_cellule_5 = instance_3sum(60, true, 500, &mut rng);

let mut accords: i64 = 0;
let mut fantomes: i64 = 0;
let mut captures_oui: i64 = 0;
for essai in 0..40 {
    let n = rng.randrange(12, 22) as usize;
    let (vals, oracle) = if essai % 2 == 0 {
        let v = instance_3sum(n, true, 120, &mut rng);
        let o = oracle_3sum_lent(&v).0;
        (v, o)
    } else {
        let v: Vec<i64> = (0..n as i64).map(|k| 3 * k + 1).collect();
        (v, false)
    };
    assert!(oracle == (essai % 2 == 0));
    let mut vu = false;
    for _ in 0..30 {
        let (a, b, c) = triparti_par_coloriage(&vals, &mut rng);
        if a.is_empty() || b.is_empty() || c.is_empty() {
            continue;
        }
        if triangle_exact_brut(&graphe_depuis_3sum_triparti(&a, &b, &c)).0 {
            vu = true;
            break;
        }
    }
    let attendu = oracle;
    if vu == attendu {
        accords += 1;
    }
    if vu && !attendu {
        fantomes += 1;
    }
    if attendu && vu {
        captures_oui += 1;
    }
}

println!("Corpus 40 instances (20 avec / 20 sans) : {} accords, {} triangle fantome", accords, fantomes);
println!("Cote 'avec triple' captures en <=30 coloriages : {}/20", captures_oui);


Corpus 40 instances (20 avec / 20 sans) : 40 accords, 0 triangle fantome


Cote 'avec triple' captures en <=30 coloriages : 20/20


In [6]:
// Mesure du 2/9 : combien de coloriages captent le triple ?
// Mt19937 : les mots MT de CPython reimplementes, pour reproduire exactement les
// tirages du carnet Python (random.Random == init_by_array([seed])).
struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }  // random.Random(seed), 0 <= seed < 2**32
    fn new_by_array(key: &[u32]) -> Mt19937 {     // init_by_array(key) de CPython
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {                         // i continue la ou la boucle precedente l'a laisse
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {              // random.random() : 53 bits
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 {  // 1 <= k <= 64 ; mots petit-boutistes, le dernier tronque
        if k <= 32 { return (self.next_u32() >> (32 - k)) as u64; }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    fn randbelow(&mut self, n: u64) -> u64 {     // n >= 1 ; rejet sur getrandbits(n.bit_length())
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randrange1(&mut self, stop: i64) -> i64 {             // randrange(n) = _randbelow(n)
        self.randbelow(stop as u64) as i64
    }
    fn sample_range(&mut self, lo: i64, hi: i64, k: usize) -> Vec<i64> {  // sample(range(lo,hi),k)
        let n = (hi - lo) as u64;
        assert!(k as u64 <= n);
        let mut result = vec![0i64; k];
        let mut setsize: u64 = 21;                            // setsize de CPython
        if k > 5 {
            setsize += 4u64.pow((((k as f64) * 3.0).log(4.0)).ceil() as u32);
        }
        if n <= setsize {
            let mut pool: Vec<i64> = (lo..hi).collect();
            for i in 0..k {
                let j = self.randbelow(n - i as u64) as usize;
                result[i] = pool[j];
                pool[j] = pool[n as usize - i - 1];
            }
        } else {
            let mut selected: std::collections::HashSet<u64> = std::collections::HashSet::new();
            for i in 0..k {
                let mut j = self.randbelow(n);
                while selected.contains(&j) { j = self.randbelow(n); }
                selected.insert(j);
                result[i] = lo + j as i64;
            }
        }
        result
    }
}

/// Instance 3SUM a n entiers dans [-borne, borne].
fn instance_3sum(n: usize, avec_solution: bool, borne: i64, rng: &mut Mt19937) -> Vec<i64> {
    let mut vals = rng.sample_range(-borne, borne + 1, n);
    if avec_solution {
        let ab = rng.sample_range(1, borne / 2, 2);
        let (a, b) = (ab[0], ab[1]);
        vals[0] = a;
        vals[1] = b;
        vals[2] = -(a + b);
    } else {
        // corpus negatif : on force un trou autour de zero parite (heuristique,
        // l'exhaustivite n'est pas requise : le controle est fait par oracle lent)
    }
    vals
}

/// Triangle Exact complet tripartite : w(AB)=a, w(BC)=b, w(CA)=c.
struct GrapheTriparti { a: Vec<i64>, b: Vec<i64>, c: Vec<i64> }

fn graphe_depuis_3sum_triparti(a: &[i64], b: &[i64], c: &[i64]) -> GrapheTriparti {
    GrapheTriparti { a: a.to_vec(), b: b.to_vec(), c: c.to_vec() }
}

fn somme_triangle(g: &GrapheTriparti, ia: usize, ib: usize, ic: usize) -> i64 {
    g.a[ia] + g.b[ib] + g.c[ic]  // w(AB)=a, w(BC)=b, w(CA)=c
}

/// Oracle brutal O(|A|.|B|.|C|) : premier triangle de somme nulle.
fn triangle_exact_brut(g: &GrapheTriparti) -> (bool, Option<(i64, i64, i64)>) {
    for ia in 0..g.a.len() {
        for ib in 0..g.b.len() {
            for ic in 0..g.c.len() {
                if somme_triangle(g, ia, ib, ic) == 0 {
                    return (true, Some((g.a[ia], g.b[ib], g.c[ic])));
                }
            }
        }
    }
    (false, None)
}

/// 3SUM mono-ensemble -> 3SUM triparti par 3-coloriage.
fn triparti_par_coloriage(nombres: &[i64], rng: &mut Mt19937) -> (Vec<i64>, Vec<i64>, Vec<i64>) {
    let mut a: Vec<i64> = Vec::new();
    let mut b: Vec<i64> = Vec::new();
    let mut c: Vec<i64> = Vec::new();
    for &v in nombres {
        let coul = rng.randrange1(3);
        if coul == 0 { a.push(v); } else if coul == 1 { b.push(v); } else { c.push(v); }
    }
    (a, b, c)
}

/// Rationnel local (fractions.Fraction n'a pas d'equivalent dans std).
struct Frac { num: i64, den: i64 }

impl Frac {
    fn new(num: i64, den: i64) -> Frac { Frac { num, den } }
    fn mul(&self, o: &Frac) -> Frac { Frac { num: self.num * o.num, den: self.den * o.den } }
    fn pow(&self, k: u32) -> Frac {
        let mut r = Frac::new(1, 1);
        for _ in 0..k { r = r.mul(self); }
        r
    }
    /// x - self
    fn sub_from_int(x: i64, o: &Frac) -> Frac { Frac { num: x * o.den - o.num, den: o.den } }
    fn to_f64(&self) -> f64 { self.num as f64 / self.den as f64 }
}

// Generateur NEUF : SEED+1, aucun rejeu des cellules precedentes.
let mut rng2 = Mt19937::new(20261111 + 1);
let mut captures: i64 = 0;
let essais: i64 = 300;
let r_coloriages: u32 = 5;
for _ in 0..essais {
    let n = 15;
    let vals = instance_3sum(n, true, 120, &mut rng2);
    let mut vu = false;
    for _ in 0..r_coloriages {
        let (a, b, c) = triparti_par_coloriage(&vals, &mut rng2);
        if !a.is_empty() && !b.is_empty() && !c.is_empty()
            && triangle_exact_brut(&graphe_depuis_3sum_triparti(&a, &b, &c)).0 {
            vu = true;
            break;
        }
    }
    if vu { captures += 1; }
}
let pr = Frac::sub_from_int(1, &Frac::new(7, 9).pow(r_coloriages));
println!("Captures en {} coloriages : {}/{} = {:.3}", r_coloriages, captures, essais, captures as f64 / essais as f64);
println!("Theorie 1-(7/9)^{} = {:.3} (le 2/9 par coloriage est la borne exacte du triple fixe)", r_coloriages, pr.to_f64());


Captures en 5 coloriages : 267/300 = 0.890


Theorie 1-(7/9)^5 = 0.715 (le 2/9 par coloriage est la borne exacte du triple fixe)


### Lecture du résultat — le pont tient par 2/9

Un coloriage capture un triple nul **fixé** avec probabilité $2/9 \approx 0{,}222$ ; cinq coloriages indépendants le capturent à $1-(7/9)^5 \approx 0{,}715$. Le carnet mesure $0{,}890$ — **au-dessus** de la borne du triple fixé, parce qu'une instance de $15$ valeurs dans $[-120, 120]$ porte souvent plusieurs triples nuls, et que le coloriage n'a besoin d'en capturer **aucun en particulier** : l'événement « capturer au moins un triple » contient celui de capturer un triple fixé. Sa probabilité est donc au moins $1-(7/9)^5$ ; cette formule est exacte pour le triple fixé. La fréquence $267/300$ est une estimation empirique, pas une borne rigoureuse sur ces probabilités. Le papier, lui, n'a pas besoin de ce pont aléatoire : ses réductions publiées ([Pǎ10, KPP16] cités p. 6) vont directement de 3SUM au **lopsided**, et sont **dérandomisées** (p. 6) — le coloriage aléatoire n'est qu'un jouet pour voir le maillon.


## 3. La mèche courte : All-Edges Sparse Triangle

**All-Edges Sparse Triangle** : un graphe creux à $m$ arêtes ; pour **chaque** arête, décide-t-elle dans un triangle ? C'est la disjointness d'ensembles **hors ligne** (papier, p. 4) : chaque sommet porte un ensemble, chaque arête pose la question « vos ensembles s'intersectent-ils ? ».

Deux bornes classiques encadraient le problème :

- **énumérer les triangles** coûte $O(m^{3/2})$ [IR78, CN85] ;
- le partage haut-degré/bas-degré [AYZ97] gagne sur les graphes denses : $O(m^{2\omega/(\omega+1)})$ — **$m^{4/3}$ même si $\omega = 2$**.

Et le piège : les instances que produisent les réductions (p. 4) ont des sommets de degré $\approx \sqrt{n}$ — sur elles, $m^{4/3} = \Theta(n^2)$ pour $\omega=2$. La borne d'énumération $m^{3/2}$ est plus grande : elle ne se confond pas avec celle du partage. Tout laissait croire le mur infranchissable : le produit matriciel rapide semblait n'être bon que pour les entrées denses, or on ne demande qu'un ensemble **creux** de sorties.

Le graphe construit ci-dessous ne contient que des arêtes entre parties externes et milieu : il est biparti, donc sans triangles. Les intersections de paires externes comptent des milieux communs et deviennent des triangles seulement si une arête externe est ajoutée.


In [7]:
// Mt19937 : les mots MT de CPython reimplementes, pour reproduire exactement les
// tirages du carnet Python (random.Random == init_by_array([seed])).
struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }  // random.Random(seed), 0 <= seed < 2**32
    fn new_by_array(key: &[u32]) -> Mt19937 {     // init_by_array(key) de CPython
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {                         // i continue la ou la boucle precedente l'a laisse
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {              // random.random() : 53 bits
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 {  // 1 <= k <= 64 ; mots petit-boutistes, le dernier tronque
        if k <= 32 { return (self.next_u32() >> (32 - k)) as u64; }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    fn randbelow(&mut self, n: u64) -> u64 {     // n >= 1 ; rejet sur getrandbits(n.bit_length())
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randrange(&mut self, start: i64, stop: i64) -> i64 {  // randrange(a, b), pas=1
        start + self.randbelow((stop - start) as u64) as i64
    }
    fn randrange1(&mut self, stop: i64) -> i64 {             // randrange(n) = _randbelow(n)
        self.randbelow(stop as u64) as i64
    }
    fn sample_range(&mut self, lo: i64, hi: i64, k: usize) -> Vec<i64> {  // sample(range(lo,hi),k)
        let n = (hi - lo) as u64;
        assert!(k as u64 <= n);
        let mut result = vec![0i64; k];
        let mut setsize: u64 = 21;                            // setsize de CPython
        if k > 5 {
            setsize += 4u64.pow((((k as f64) * 3.0).log(4.0)).ceil() as u32);
        }
        if n <= setsize {
            let mut pool: Vec<i64> = (lo..hi).collect();
            for i in 0..k {
                let j = self.randbelow(n - i as u64) as usize;
                result[i] = pool[j];
                pool[j] = pool[n as usize - i - 1];
            }
        } else {
            let mut selected: std::collections::HashSet<u64> = std::collections::HashSet::new();
            for i in 0..k {
                let mut j = self.randbelow(n);
                while selected.contains(&j) { j = self.randbelow(n); }
                selected.insert(j);
                result[i] = lo + j as i64;
            }
        }
        result
    }
}

/// Instance 3SUM a n entiers dans [-borne, borne].
fn instance_3sum(n: usize, avec_solution: bool, borne: i64, rng: &mut Mt19937) -> Vec<i64> {
    let mut vals = rng.sample_range(-borne, borne + 1, n);
    if avec_solution {
        let ab = rng.sample_range(1, borne / 2, 2);
        let (a, b) = (ab[0], ab[1]);
        vals[0] = a;
        vals[1] = b;
        vals[2] = -(a + b);
    } else {
        // corpus negatif : on force un trou autour de zero parite (heuristique,
        // l'exhaustivite n'est pas requise : le controle est fait par oracle lent)
    }
    vals
}

/// Oracle O(n^3) : triple distinct de somme nulle.
#[allow(dead_code)]
fn oracle_3sum_lent(nombres: &[i64]) -> (bool, Option<(i64, i64, i64)>) {
    let n = nombres.len();
    for i in 0..n {
        for j in (i + 1)..n {
            for k in (j + 1)..n {
                if nombres[i] + nombres[j] + nombres[k] == 0 {
                    return (true, Some((nombres[i], nombres[j], nombres[k])));
                }
            }
        }
    }
    (false, None)
}

/// Triangle Exact complet tripartite : w(AB)=a, w(BC)=b, w(CA)=c.
struct GrapheTriparti { a: Vec<i64>, b: Vec<i64>, c: Vec<i64> }

fn graphe_depuis_3sum_triparti(a: &[i64], b: &[i64], c: &[i64]) -> GrapheTriparti {
    GrapheTriparti { a: a.to_vec(), b: b.to_vec(), c: c.to_vec() }
}

fn somme_triangle(g: &GrapheTriparti, ia: usize, ib: usize, ic: usize) -> i64 {
    g.a[ia] + g.b[ib] + g.c[ic]  // w(AB)=a, w(BC)=b, w(CA)=c
}

/// Oracle brutal O(|A|.|B|.|C|) : premier triangle de somme nulle.
fn triangle_exact_brut(g: &GrapheTriparti) -> (bool, Option<(i64, i64, i64)>) {
    for ia in 0..g.a.len() {
        for ib in 0..g.b.len() {
            for ic in 0..g.c.len() {
                if somme_triangle(g, ia, ib, ic) == 0 {
                    return (true, Some((g.a[ia], g.b[ib], g.c[ic])));
                }
            }
        }
    }
    (false, None)
}

/// 3SUM mono-ensemble -> 3SUM triparti par 3-coloriage.
fn triparti_par_coloriage(nombres: &[i64], rng: &mut Mt19937) -> (Vec<i64>, Vec<i64>, Vec<i64>) {
    let mut a: Vec<i64> = Vec::new();
    let mut b: Vec<i64> = Vec::new();
    let mut c: Vec<i64> = Vec::new();
    for &v in nombres {
        let coul = rng.randrange1(3);
        if coul == 0 { a.push(v); } else if coul == 1 { b.push(v); } else { c.push(v); }
    }
    (a, b, c)
}

/// All-Edges Sparse Triangle par appartenance : pour chaque arete (u,v),
/// u et v ont-ils un voisin commun ? (disjointness hors ligne.)
/// graphe_adj : sommet -> ensemble de voisins. Retourne arete -> bool.
/// Code mort dans le carnet Python (defini, jamais appele) : defini ici pour la
/// compilation et le fil pedagogique, sans l'appeler.
#[allow(dead_code)]
fn toutes_aretes_triangle(graphe_adj: &Vec<BTreeSet<usize>>) -> BTreeMap<(usize, usize), bool> {
    let mut res: BTreeMap<(usize, usize), bool> = BTreeMap::new();
    let mut vues: std::collections::HashSet<(usize, usize)> = std::collections::HashSet::new();
    for (u, vois) in graphe_adj.iter().enumerate() {
        for &v in vois {
            let arete = (u.min(v), u.max(v));
            if vues.contains(&arete) {
                continue;
            }
            vues.insert(arete);
            res.insert(arete, vois.intersection(&graphe_adj[v]).count() > 0);
        }
    }
    res
}

use std::collections::{BTreeMap, BTreeSet};
use std::time::Instant;

/// Graphe tripartite lopsided : parties P1, P2 de N sommets,
/// partie moyenne M de D sommets. Adjacence P-M creuse.
fn evalue_graphe_lopsided(n: usize, d: usize, densite: f64, rng: &mut Mt19937) -> Vec<BTreeSet<usize>> {
    let mut adj: Vec<BTreeSet<usize>> = Vec::new();
    for _ in 0..(2 * n + d) {
        adj.push(BTreeSet::new());
    }
    for p in 0..(2 * n) {
        for m in (2 * n)..(2 * n + d) {
            if rng.random() < densite {
                adj[p].insert(m);
                adj[m].insert(p);
            }
        }
    }
    adj
}

// Sous evcxr rien ne persiste : on rejoue la cellule 5 (66 mots) puis la cellule 8
// (corpus complet, casses precoces comprises) sur un generateur neuf, sans imprimer.
let mut rng = Mt19937::new(20261111);
let _rejeu_cellule_5 = instance_3sum(60, true, 500, &mut rng);
for essai in 0..40 {
    let n = rng.randrange(12, 22) as usize;
    let vals: Vec<i64> = if essai % 2 == 0 {
        instance_3sum(n, true, 120, &mut rng)
    } else {
        (0..n as i64).map(|k| 3 * k + 1).collect()
    };
    let mut vu = false;
    for _ in 0..30 {
        let (a, b, c) = triparti_par_coloriage(&vals, &mut rng);
        if a.is_empty() || b.is_empty() || c.is_empty() {
            continue;
        }
        if triangle_exact_brut(&graphe_depuis_3sum_triparti(&a, &b, &c)).0 {
            vu = true;
            break;
        }
    }
    let _ = vu;
}

// Les paires externes comptent des milieux communs ; les seules aretes P-M
// ne ferment aucun triangle dans ce graphe biparti.
let n_jouet: usize = 40;
let d_jouet: usize = 6;
let dens: f64 = 0.55;
let adj = evalue_graphe_lopsided(n_jouet, d_jouet, dens, &mut rng);
let t0 = Instant::now();
let mut inter: BTreeMap<(usize, usize), usize> = BTreeMap::new();
for p1 in 0..n_jouet {
    for p2 in n_jouet..(2 * n_jouet) {
        let commun = adj[p1]
            .intersection(&adj[p2])
            .filter(|&&v| v >= 2 * n_jouet)
            .count();
        inter.insert((p1, p2), commun);
    }
}
let t1 = Instant::now();
let nb_pos = inter.values().filter(|&&v| v > 0).count();
println!("Graphe lopsided N={}, D={}, densite={}", n_jouet, d_jouet, dens);
println!("Paires externes = {} = N^2 ; paires avec voisin commun : {}", inter.len(), nb_pos);
println!("Temps produit-complet des intersections : {:.2} ms", (t1 - t0).as_secs_f64() * 1000.0);


Graphe lopsided N=40, D=6, densite=0.55


Paires externes = 1600 = N^2 ; paires avec voisin commun : 1449


Temps produit-complet des intersections : 0.38 ms


### Lecture du résultat — $m^{4/3}$ sur le graphe de réduction

Pour $n$ sommets de degré moyen $\sqrt{n}$, la somme des degrés vaut $2m$, donc $m = n^{3/2}/2$. Les deux bornes donnent alors :

$$m^{3/2} = n^{9/4}/2^{3/2}, \qquad m^{4/3} = n^2/2^{4/3}.$$

L'énumération a donc un exposant $9/4$ : elle dépasse le mur quadratique asymptotiquement. Le partage AYZ, lui, donne $\Theta(n^2)$ lorsque $\omega=2$ ; les constantes ne changent pas cet ordre. C'est ce verrou que le papier fait sauter pour les instances **lopsided**.


## 4. La version lopsided et le produit matriciel fin

Le moteur (papier, p. 5-6) : soit $X \in \mathbb{Z}^{N\times D}$ et $Y \in \mathbb{Z}^{D\times N}$ avec $D \le N^{1/18}$ (un produit « **fin** »), et un ensemble $W$ d'au plus $N^2/\sqrt{D}$ positions « **voulues** ». Alors les entrées $(XY)[I,J]$ pour $(I,J) \in W$ se calculent en

$$O\left(N^2 / D^{0{,}063}\right) \text{ opérations}$$

— polynômialement **moins** que le temps d'écrire $XY$ entier ($\Omega(N^2)$) ou de calculer $|W|$ produits scalaires un à un. Lue en graphe : $X$ et $Y$ sont les biadjacences vers la partie moyenne, $(XY)[a,b]$ **compte** les milieux communs — la version comptage de Lopsided All-Edges Sparse Triangle, qui décide la détection.

Sur un graphe tripartite à deux parties de $n$ sommets et une partie moyenne de $n^{\varepsilon}$ : compter les triangles à travers $|W|$ paires prescrites en $O(|W|\,n^{0{,}437\varepsilon} + n^{2-0{,}063\varepsilon})$ pour $\varepsilon < 1/18$ — vraiment sous-quadratique pour tout $\varepsilon < 0{,}1204$.


In [8]:
/// Nombre de produits scalaires du produit COMPLET : N^2 produits de longueur D.
fn produit_complet_ops(n: i64, d: i64, _w: Option<i64>) -> i64 {
    n * n * d
}

/// Produits scalaires du calcul NAIF voulu : |W| produits de longueur D.
/// (Le papier fait mieux : O(N^2 / D^0.063) — compare les deux regimes.)
fn produit_voulu_ops(_n: i64, _d: i64, taille_w: i64) -> i64 {
    taille_w * _d
}

// Comparaison de formules pour N croissant ; voir le domaine admissible ci-dessous
println!("{:>5} {:>7} {:>17} {:>11} {:>13} {:>12}",
         "N", "D=N^e", "|W|<=N^2/sqrt(D)", "naif |W|.D", "complet N^2.D", "N^2/D^0.063");
for n in [64i64, 128i64, 256i64, 512i64, 1024i64].iter().copied() {
    let d = std::cmp::max(2, ((n as f64).powf(1.0 / 18.0)).round() as i64);  // ici D=2, hors domaine pour ces N
    let borne_w = ((n * n) as f64 / (d as f64).sqrt()) as i64;  // plafond autorise sur |W|
    println!("{:>5} {:>7} {:>17} {:>11} {:>13} {:>12.0}",
             n, d, borne_w,
             produit_voulu_ops(n, d, borne_w),
             produit_complet_ops(n, d, None),
             (n * n) as f64 / (d as f64).powf(0.063));
}

let _ = ();

    N   D=N^e  |W|<=N^2/sqrt(D)  naif |W|.D complet N^2.D  N^2/D^0.063


   64       2              2896        5792          8192         3921


  128       2             11585       23170         32768        15684


  256       2             46340       92680        131072        62736


  512       2            185363      370726        524288       250943


 1024       2            741455     1482910       2097152      1003772


### Lecture du résultat — trois régimes, un seul gagnant

Dans le domaine $D \le N^{1/18}$, prenons $|W|$ au plafond $N^2/\sqrt{D}$. Le calcul naïf voulu coûte $|W|D=N^2D^{1/2}$, tandis que le produit complet coûte $N^2D$. Le rapport entre le premier et l'expression de la borne est

$$\frac{N^2D^{1/2}}{N^2/D^{0{,}063}}=D^{0{,}563}.$$

C'est une comparaison d'ordres asymptotiques : la constante cachée dans $O(\cdot)$ empêche d'en déduire un facteur de temps effectif pour une instance finie. Pour $D=N^{1/18}$, ce rapport croît polynomialement en $N$. Le cœur technique modifie l'algorithme rectangulaire de Coppersmith (identité à dix multiplications de Schönhage, p. 6) pour ne calculer que les entrées voulues.

Le tableau est une comparaison **extrapolée hors du domaine** : toutes ses lignes ont $D=2$ et $N<2^{18}$, donc $2>N^{1/18}$. Pour $D=2$, l'hypothèse exige $N\ge 2^{18}$ ; les nombres du tableau ne démontrent donc pas l'application du théorème à ces instances. L'arrondi de $|W|$ explique les petits écarts aux expressions continues.

À $D=2$, ces expressions valent environ $1{,}414N^2$ pour le calcul naïf et $0{,}957N^2$ pour la borne, soit un rapport d'environ $1{,}48$. Il dépasse $2$ lorsque $D>2^{1/0{,}563}\approx 3{,}43$. À $D=5$, la condition du théorème demande $N\ge 5^{18}$, soit plus de $10^{12}$. Comme dans le [03e](Complexity-03e-Barrieres-Exposants-WordRAM-Python.ipynb), ces comparaisons d'exposants ne sont pas des chronomètres.


## 5. Ce qui survit, ce qui tombe

Le verdict d'octobre 2026 (Théorème 2, p. 7 ; OMv Théorème 4, p. 8) :

| Objet | Statut après le papier |
|---|---|
| Hypothèse 3SUM | **réfutée** — déterministe $O(n^{1{,}9992})$ word RAM ; $O(n^{1{,}998})$ Las Vegas en RAM réelle |
| Hypothèse APSP | **réfutée** — $O(n^{2{,}9995})$ (produit (min,+) idem) |
| Triangle Exact | $O(n^{2{,}9983})$ déterministe ; $O(n^{2{,}998})$ Las Vegas |
| k-Clique de poids nul | $O(n^{k-\varepsilon})$ pour un $\varepsilon > 0$ |
| Conjectures OMv « à indice fin » | **réfutées** pour $\tau < 1/18$ (précalcul $O(n^{2-0{,}063\tau})$) |
| Bornes dynamiques principales (inverse matriciel dynamique) | **intactes** — elles utilisent $\tau \approx 0{,}53$, loin du régime fin |
| **SETH**, Orthogonal Vectors | **intactes** — pas dans la toile du papier |
| Arbres de décision 3-linéaires pour 3SUM : $\Omega(n^2)$ | **intactes** [Eri99, AC05] — le nouvel algorithme sort du modèle |
| Programmes droits sur le semi-anneau (min,+) : $n^3$ | **intacts** [Ker70] — sortie du modèle par hachage des poids |

La sortie des modèles restreints (p. 7) : les réductions connues **retirent les poids** (hachage, ou l'astuce de Fredman pour les entrées réelles), puis comptent des triangles de graphe **non pondéré** par produit matriciel entier. Le comptage est plus fort que la détection — et la classe APSP entière suit par la toile.


In [9]:
// Lecture des verdicts : distinguer les réfutations établies par le papier des
// conjectures qui restent intactes parce qu'elles ne relèvent pas de la toile.
// La colonne « Algo / statut » rappelle brièvement le résultat associé à chaque objet.
let verdicts: [(&str, &str, &str); 10] = [
    ("3SUM", "n^1.9992 det / n^1.998 LV", "REFUTEE"),
    ("APSP", "n^2.9995 det / n^2.998 LV", "REFUTEE"),
    ("Triangle Exact", "n^2.9983 det", "REFUTEE (a sa dureté conditionnelle)"),
    ("(min,+)-produit", "n^2.9995 det", "REFUTEE"),
    ("k-Clique poids nul", "n^(k-eps)", "REFUTEE"),
    ("OMv indice fin (tau<1/18)", "precalc n^2-0.063tau", "REFUTEE"),
    ("SETH", "—", "INTACTE"),
    ("Orthogonal Vectors", "—", "INTACTE"),
    ("3-LDT 3SUM Omega(n^2)", "hors du modele", "INTACTE (modele restreint)"),
    ("SLP (min,+) n^3", "hors du modele", "INTACTE (modele restreint)"),
];

// En-tête commun, puis séparation visuelle avant le tableau des verdicts.
println!("{:<28} {:<26} Verdict", "Objet", "Algo / statut");
println!("{}", "-".repeat(78));

// Chaque triplet contient l'objet, l'algorithme ou le statut, puis le verdict.
for (obj, algo, verdict) in verdicts.iter() {
    println!("{:<28} {:<26} {}", obj, algo, verdict);
}

let _ = ();

Objet                        Algo / statut              Verdict


------------------------------------------------------------------------------


3SUM                         n^1.9992 det / n^1.998 LV  REFUTEE


APSP                         n^2.9995 det / n^2.998 LV  REFUTEE


Triangle Exact               n^2.9983 det               REFUTEE (a sa dureté conditionnelle)


(min,+)-produit              n^2.9995 det               REFUTEE


... (2 outputs left out)

SETH                         —                          INTACTE


Orthogonal Vectors           —                          INTACTE


3-LDT 3SUM Omega(n^2)        hors du modele             INTACTE (modele restreint)


SLP (min,+) n^3              hors du modele             INTACTE (modele restreint)


## 6. Exercices

Trois exercices, à compléter sans casser l'exécution (les stubs retournent `None` ; le carnet reste exécutable de bout en bout).


### Exercice 1 — le côté négatif du pont

Le corpus « sans solution » de la section 2 était contrôlé par l'oracle lent. Étendez la vérification : générez 10 instances **sans** triple nul (oracle lent d'accord), passez-les par 30 coloriages chacune, et vérifiez qu'**aucune** ne produit de triangle de somme nulle — le coloriage ne peut pas inventer un triple. Retournez le nombre d'instances testées et le nombre de triangles fantômes (attendu : 0).


In [10]:
/// Retourne (nb_testees, nb_triangles_fantomes) -- attendu (10, 0).
/// TODO etudiant
fn exercice1_sans_solution() -> Option<(usize, usize)> {
    // Etape 1 : generer nb_instances instances SANS triple nul (oracle_3sum_lent)
    // Etape 2 : pour chaque, nb_coloriages coloriages -> graphe triparti
    // Etape 3 : compter les triangles de somme nulle (triangle_exact_brut)
    // Indice : un graphe complet tripartite depuis des valeurs sans triple nul
    //          ne peut pas produire de somme nulle -- prouvez-le par la mesure.
    println!("Exercice 1 a completer");
    None  // TODO etudiant
}

/// Rendu de None a la maniere Python.
fn py_opt<T>(v: &Option<T>) -> String {
    match v {
        Some(_) => "Some(...)".to_string(),
        None => "None".to_string(),
    }
}

let r1 = exercice1_sans_solution();
println!("Exercice 1 : {}", py_opt(&r1));


Exercice 1 a completer


Exercice 1 : None


### Exercice 2 — All-Edges Sparse Triangle par énumération

Implémentez le comptage **par arête** sur le graphe lopsided de la section 3 : pour chaque arête $(u,v)$, le nombre de triangles qui la contiennent (pas seulement oui/non). Combien d'arêtes sont dans au moins un triangle ?

Le graphe généré est biparti : il ne contient aucun triangle. Vérifiez aussi votre fonction sur un graphe contenant un triangle et un sommet isolé, afin de détecter une réponse toujours nulle.


In [11]:
use std::collections::{BTreeMap, BTreeSet};

// Mt19937 : les mots MT de CPython reimplementes, pour reproduire exactement les
// tirages du carnet Python (random.Random == init_by_array([seed])).
struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }  // random.Random(seed), 0 <= seed < 2**32
    fn new_by_array(key: &[u32]) -> Mt19937 {     // init_by_array(key) de CPython
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {                         // i continue la ou la boucle precedente l'a laisse
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {              // random.random() : 53 bits
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 {  // 1 <= k <= 64 ; mots petit-boutistes, le dernier tronque
        if k <= 32 { return (self.next_u32() >> (32 - k)) as u64; }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    fn randbelow(&mut self, n: u64) -> u64 {     // n >= 1 ; rejet sur getrandbits(n.bit_length())
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randrange(&mut self, start: i64, stop: i64) -> i64 {  // randrange(a, b), pas=1
        start + self.randbelow((stop - start) as u64) as i64
    }
    fn randrange1(&mut self, stop: i64) -> i64 {             // randrange(n) = _randbelow(n)
        self.randbelow(stop as u64) as i64
    }
    fn sample_range(&mut self, lo: i64, hi: i64, k: usize) -> Vec<i64> {  // sample(range(lo,hi),k)
        let n = (hi - lo) as u64;
        assert!(k as u64 <= n);
        let mut result = vec![0i64; k];
        let mut setsize: u64 = 21;                            // setsize de CPython
        if k > 5 {
            setsize += 4u64.pow((((k as f64) * 3.0).log(4.0)).ceil() as u32);
        }
        if n <= setsize {
            let mut pool: Vec<i64> = (lo..hi).collect();
            for i in 0..k {
                let j = self.randbelow(n - i as u64) as usize;
                result[i] = pool[j];
                pool[j] = pool[n as usize - i - 1];
            }
        } else {
            let mut selected: std::collections::HashSet<u64> = std::collections::HashSet::new();
            for i in 0..k {
                let mut j = self.randbelow(n);
                while selected.contains(&j) { j = self.randbelow(n); }
                selected.insert(j);
                result[i] = lo + j as i64;
            }
        }
        result
    }
}

/// Instance 3SUM a n entiers dans [-borne, borne].
fn instance_3sum(n: usize, avec_solution: bool, borne: i64, rng: &mut Mt19937) -> Vec<i64> {
    let mut vals = rng.sample_range(-borne, borne + 1, n);
    if avec_solution {
        let ab = rng.sample_range(1, borne / 2, 2);
        let (a, b) = (ab[0], ab[1]);
        vals[0] = a;
        vals[1] = b;
        vals[2] = -(a + b);
    } else {
        // corpus negatif : on force un trou autour de zero parite (heuristique,
        // l'exhaustivite n'est pas requise : le controle est fait par oracle lent)
    }
    vals
}

/// Triangle Exact complet tripartite : w(AB)=a, w(BC)=b, w(CA)=c.
struct GrapheTriparti { a: Vec<i64>, b: Vec<i64>, c: Vec<i64> }

fn graphe_depuis_3sum_triparti(a: &[i64], b: &[i64], c: &[i64]) -> GrapheTriparti {
    GrapheTriparti { a: a.to_vec(), b: b.to_vec(), c: c.to_vec() }
}

fn somme_triangle(g: &GrapheTriparti, ia: usize, ib: usize, ic: usize) -> i64 {
    g.a[ia] + g.b[ib] + g.c[ic]  // w(AB)=a, w(BC)=b, w(CA)=c
}

/// Oracle brutal O(|A|.|B|.|C|) : premier triangle de somme nulle.
fn triangle_exact_brut(g: &GrapheTriparti) -> (bool, Option<(i64, i64, i64)>) {
    for ia in 0..g.a.len() {
        for ib in 0..g.b.len() {
            for ic in 0..g.c.len() {
                if somme_triangle(g, ia, ib, ic) == 0 {
                    return (true, Some((g.a[ia], g.b[ib], g.c[ic])));
                }
            }
        }
    }
    (false, None)
}

/// 3SUM mono-ensemble -> 3SUM triparti par 3-coloriage.
fn triparti_par_coloriage(nombres: &[i64], rng: &mut Mt19937) -> (Vec<i64>, Vec<i64>, Vec<i64>) {
    let mut a: Vec<i64> = Vec::new();
    let mut b: Vec<i64> = Vec::new();
    let mut c: Vec<i64> = Vec::new();
    for &v in nombres {
        let coul = rng.randrange1(3);
        if coul == 0 { a.push(v); } else if coul == 1 { b.push(v); } else { c.push(v); }
    }
    (a, b, c)
}

/// Graphe tripartite lopsided : parties P1, P2 de N sommets,
/// partie moyenne M de D sommets. Adjacence P-M creuse.
fn evalue_graphe_lopsided(n: usize, d: usize, densite: f64, rng: &mut Mt19937) -> Vec<BTreeSet<usize>> {
    let mut adj: Vec<BTreeSet<usize>> = Vec::new();
    for _ in 0..(2 * n + d) {
        adj.push(BTreeSet::new());
    }
    for p in 0..(2 * n) {
        for m in (2 * n)..(2 * n + d) {
            if rng.random() < densite {
                adj[p].insert(m);
                adj[m].insert(p);
            }
        }
    }
    adj
}

/// Retourne dict arete -> nb de triangles la contenant.
/// TODO etudiant
fn exercice2_triangles_par_arete(adj: &Vec<BTreeSet<usize>>) -> Option<BTreeMap<(usize, usize), usize>> {
    // Etape 1 : pour chaque arete (u,v), intersector les voisinages
    // Etape 2 : chaque voisin commun w forme le triangle (u,v,w)
    // Indice : l'arete P1-P2 n'existe pas dans le biparti -- seules les aretes
    //          vers la partie moyenne (et d'eventuelles aretes P-P) comptent.
    let _ = adj;
    println!("Exercice 2 a completer");
    None  // TODO etudiant
}

/// Rendu de None a la maniere Python.
fn py_opt<T>(v: &Option<T>) -> String {
    match v {
        Some(_) => "Some(...)".to_string(),
        None => "None".to_string(),
    }
}

// Sous evcxr rien ne persiste : on reconstruit le graphe du carnet par le meme rejeu
// que la cellule 12 (cellules 5 et 8, puis les 480 tirages du graphe lopsided).
let mut rng = Mt19937::new(20261111);
let _rejeu_cellule_5 = instance_3sum(60, true, 500, &mut rng);
for essai in 0..40 {
    let n = rng.randrange(12, 22) as usize;
    let vals: Vec<i64> = if essai % 2 == 0 {
        instance_3sum(n, true, 120, &mut rng)
    } else {
        (0..n as i64).map(|k| 3 * k + 1).collect()
    };
    let mut vu = false;
    for _ in 0..30 {
        let (a, b, c) = triparti_par_coloriage(&vals, &mut rng);
        if a.is_empty() || b.is_empty() || c.is_empty() {
            continue;
        }
        if triangle_exact_brut(&graphe_depuis_3sum_triparti(&a, &b, &c)).0 {
            vu = true;
            break;
        }
    }
    let _ = vu;
}
let adj = evalue_graphe_lopsided(40, 6, 0.55, &mut rng);

let r2 = exercice2_triangles_par_arete(&adj);
println!("Exercice 2 : {}", py_opt(&r2));


Exercice 2 a completer


Exercice 2 : None


### Exercice 3 — le régime du wanted set

Pour $\varepsilon \in \{1/36, 1/18\}$ et $N \in \{2^6, 2^8, 2^{10}\}$, posez formellement $D=N^\varepsilon$ et calculez $|W|_{\max}/N^2$, puis le rapport entre le coût naïf $|W|_{\max}D$ et l'expression $N^2/D^{0{,}063}$. À partir de quel $D$ ce rapport dépasse-t-il 2 ?

Pour interpréter les résultats, distinguez cette comparaison de formules d'un temps mesuré. $D$ doit être entier pour une dimension matricielle : discutez l'effet d'un arrondi et vérifiez à nouveau $D \le N^{1/18}$. Le plafond sur $|W|$ est une hypothèse du résultat cité ; il ne dit pas à lui seul ce qui se passe au-dessus.


In [12]:
/// Retourne la liste [(eps, N, rapport_W_sur_N2, gain_borne), ...].
/// TODO etudiant
fn exercice3_regime_wanted() -> Option<Vec<(f64, i64, f64, f64)>> {
    // Etape 1 : D = N^eps ; W_max = N^2 / sqrt(D)
    // Etape 2 : naif = W_max * D ; borne = N^2 / D^0.063 ; gain = naif / borne
    // Indice : gain > 2 des que D^0.563 > 2 -- dans quel regime de eps ?
    println!("Exercice 3 a completer");
    None  // TODO etudiant
}

/// Rendu de None a la maniere Python.
fn py_opt<T>(v: &Option<T>) -> String {
    match v {
        Some(_) => "Some(...)".to_string(),
        None => "None".to_string(),
    }
}

let r3 = exercice3_regime_wanted();
println!("Exercice 3 : {}", py_opt(&r3));


Exercice 3 a completer


Exercice 3 : None


## Conclusion

| Maillon | Ce qu'on a fait | Mesure |
|---|---|---|
| Mur 3SUM | double balayage sur instance sans solution | $\approx n^2/2$ opérations |
| Pont coloriage | 3SUM mono-ensemble → triparti | $2/9$ par coloriage, $1-(7/9)^5$ à cinq essais |
| Réduction 3SUM → Triangle Exact | formes linéaires $w(AB)=a$ etc. | accords sur le corpus |
| Barrière $m^{4/3}$ | AYZ97 = mur sur instances de réduction (degré $\sqrt{n}$) | ordre de grandeur comparé |
| Moteur lopsided | produit fin, entrées voulues vs complet | $|W| \cdot D$ vs $N^2 D$ vs $N^2/D^{0{,}063}$ |
| Toile après octobre 2026 | conjectures réfutées par le papier et verdicts préservés hors de son modèle | la chaîne distingue les gains obtenus des limites du modèle |

## À retenir

1. **Une réduction est un certificat à double emploi.** Une réduction de 3SUM vers Triangle Exact transmet la dureté de 3SUM au Triangle Exact et permet aussi de conclure « 3SUM est rapide dès que Triangle Exact est rapide » — la toile de dureté est le conduit de l'accélération (p. 5 : *la toile interprétée comme dureté est exactement ce qui transforme un algorithme en une myriade*).
2. **La construction est des formes linéaires.** $w(u^A_a, u^B_b) = a$, $w(u^B_b, u^C_c) = b$, $w(u^C_c, u^A_a) = c$ : la somme du triangle est $a+b+c$, le triangle nul **est** le triple nul — vérifié sur corpus, 0 fantôme sur le côté négatif.
3. **Le verrou était structurel, pas numérique.** Sur les instances de réduction (degré $\approx\sqrt{n}$), la borne d’énumération $m^{3/2}$ est en $\Theta(n^{9/4})$, tandis que le partage AYZ $m^{4/3}$ est en $\Theta(n^2)$ pour $\omega=2$ : la sortie creuse semblait hors de portée du produit matriciel. Le moteur fin ($D \le N^{1/18}$, $|W| \le N^2/\sqrt{D}$, $O(N^2/D^{0{,}063})$) calcule **moins** que ce qu'il faudrait pour écrire le produit.
4. **Réfuter une hypothèse ne refute pas le domaine.** SETH et Orthogonal Vectors restent debout, les bornes en modèles restreints (3-linéaires, programmes droits, comparaison de chemins) restent vraies — le nouvel algorithme **sort** de ces modèles (hachage des poids, comptage entier sur graphes non pondérés).


## Vérifiez votre compréhension

1. Pourquoi l'instance **sans** solution coûte-t-elle $(n-1)(n-2)/2$ comparaisons au double balayage, et pas l'instance **avec** ?
2. Le coloriage aléatoire capture un triple nul donné avec probabilité $2/9$ — d'où sortent le 6 et le 27 du calcul ?
3. Sur une instance de réduction à degré $\approx\sqrt{n}$, montrez que $m^{4/3} \approx n^2$ (combien d'arêtes $m$ a-t-elle ?).
4. À $D = N^{1/18}$, pourquoi $N^2/D^{0{,}063}$ est-il **polynômialement** en dessous de $|W| \cdot D$ pris à $|W| = N^2/\sqrt{D}$ — et pourquoi le tableau à $N = 1024$ ne permet-il pas de conclure à un gain mesuré ?

<details><summary>Réponses</summary>

1. Chaque ancre sans solution parcourt toutes les positions des deux pointeurs, soit $(n-1)(n-2)/2$ ; une solution peut interrompre le parcours. 2. Il y a $3^3=27$ coloriages et $3!=6$ affectations bijectives. 3. La somme des degrés donne $m=\Theta(n^{3/2})$, donc $m^{4/3}=\Theta(n^2)$. 4. Le quotient des expressions vaut $D^{0,563}$ ; les dimensions du tableau sont hors domaine et les constantes cachées ne sont pas des temps mesurés.
</details>


## Pour aller plus loin

- **Carnet jumeau** : [Complexity-03e — barrières d'exposants, word RAM](Complexity-03e-Barrieres-Exposants-WordRAM-Python.ipynb) — les cinq claims vérifiés à la machine, le témoin fermé, le facteur $n^{0{,}0008}$ calculé.
- **L'autre annonce vérifiée** : [Complexity-03d — la réfutation annoncée de l'ETH](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb) — pourquoi elle, en revanche, attend toujours sa preuve.
- **Le panorama** : [Complexity-07c — le bilan mathocalypse](Complexity-07c-LeBilanMathocalypse-Python.ipynb) — où s'inscrit le contrepoint Williams–Alman du 5 octobre.
- **Carnet suivant de la série** : [Complexity-04 — algorithmes en ligne](Complexity-04-OnlineAlgorithms-Python.ipynb) — quitter la toile d'hypothèses pour les décisions sans connaître la suite.
- **Source** : Josh Alman & Virginia Vassilevska Williams, *Truly Subquadratic 3SUM and Truly Subcubic APSP via Triangles in Sparse Lopsided Graphs*, arXiv [2610.06783](https://arxiv.org/abs/2610.06783) (10/2026, 76 p.) — copie archivée au gisement `Bibliographie IA/Automata/Complexity Theory and Hardness`. Toute attribution du carnet se lit contre cette copie.

**Navigation** : [Index](README.md) | [Version C#](Complexity-03f-ReductionsTriangle-Lopsided-CSharp.ipynb) | [Version Python](Complexity-03f-ReductionsTriangle-Lopsided-Python.ipynb)

> Carnet issu de l'EPIC #26534.
